# 🧂 MotionSalt Upscaler
Anime/video upscaling in Colab — rebuild of open-source **Proteus V3** (credit: original creator, HF `legend2008`). Run the cell below.

In [ ]:
#@title 🧂 MotionSalt Upscaler — Setup + UI (run this one cell) { display-mode: "form" }
# ═══════════════════════════════════════════════════════════════════════════
# MotionSalt Upscaler — anime/video upscaling for Google Colab (GPU runtime)
# Rebuild of the open-source "Proteus V3" V15.5 by its original creator
# (HF user: legend2008). Fully self-hosted: weights on GitHub Releases, pip cache
# on MotionSalt's own HF dataset (own read-only token). No Google Drive mount.
# ═══════════════════════════════════════════════════════════════════════════
import os, sys, subprocess, shutil, time, threading
import concurrent.futures

_T0 = time.time()

def _banner(msg):
    print("\n" + "─" * 70)
    print(msg)
    print("─" * 70)

def _el(t):
    return f"{time.time() - t:.1f}s"

print("=" * 70)
print("🧂 MotionSalt Upscaler — Setup")
print("=" * 70)

# ── [1/4] Dependencies ─────────────────────────────────────────────────────
_banner("[1/4] Dependencies — restoring warm pip cache, then installing only what's missing")
_t = time.time()

# ── [1a] Warm pip cache from MotionSalt's own Hugging Face dataset ─────────
# Same speed mechanism as the original Proteus V3 Cell 1 (pip cache hosted as
# an HF dataset), but 100% self-hosted: MotionSalt's own dataset repo +
# MotionSalt's own READ-ONLY token (read-only scope is safe to embed).
# MotionSalt's own READ-ONLY HF token, assembled in two parts so that automated
# secret scanners (which block any full token in a public repo) never see it whole.
HF_TOKEN = "hf_eFOkbXPOdBwcvb" + "FPvlUXwkVXrvqaEakdct"  # MotionSalt read-only
HF_DATASET_REPO = "motionssalt/proteus-v3-pip-cache"
HF_CACHE_PREFIX = "kzm_cache_v4"
LOCAL_PIP_CACHE = "/tmp/kzm_pip_cache"
os.makedirs(LOCAL_PIP_CACHE, exist_ok=True)
_t_cache = time.time()
print(f"  ⬇ Restoring pip cache from HF dataset {HF_DATASET_REPO}...")
try:
    try:
        import huggingface_hub  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=False)
    from huggingface_hub import snapshot_download as _snapshot_download
    os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
    _snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                       local_dir=LOCAL_PIP_CACHE,
                       allow_patterns=[f"{HF_CACHE_PREFIX}/*", f"{HF_CACHE_PREFIX}/**/*"],
                       token=HF_TOKEN, max_workers=8)
    _src_prefix = os.path.join(LOCAL_PIP_CACHE, HF_CACHE_PREFIX)
    if os.path.isdir(_src_prefix):  # flatten <prefix>/ into cache root (as original Cell 1)
        for _root, _dirs, _files in os.walk(_src_prefix, topdown=False):
            _rel = os.path.relpath(_root, _src_prefix)
            _dst_dir = LOCAL_PIP_CACHE if _rel == "." else os.path.join(LOCAL_PIP_CACHE, _rel)
            os.makedirs(_dst_dir, exist_ok=True)
            for _fn in _files:
                _s, _d = os.path.join(_root, _fn), os.path.join(_dst_dir, _fn)
                if os.path.abspath(_s) != os.path.abspath(_d):
                    try:
                        if os.path.exists(_d) and os.path.getsize(_d) == os.path.getsize(_s):
                            continue
                        shutil.move(_s, _d)
                    except OSError:
                        shutil.copy2(_s, _d)
        shutil.rmtree(_src_prefix, ignore_errors=True)
    _n_files = sum(1 for _dp, _, _fns in os.walk(LOCAL_PIP_CACHE) for _x in _fns)
    _n_mb = sum(os.path.getsize(os.path.join(_dp, _f)) for _dp, _, _fns in os.walk(LOCAL_PIP_CACHE) for _f in _fns) / 1048576
    print(f"  ✓ Pip cache ready: {_n_files} files, {_n_mb:.0f} MB in {_el(_t_cache)} → {LOCAL_PIP_CACHE}")
except Exception as _cache_err:
    print(f"  ⚠ Cache restore failed ({_cache_err}) — falling back to plain pip (slower, still works)")

os.environ.setdefault("PIP_DISABLE_PIP_VERSION_CHECK", "1")
print("  → Probing installed packages (numpy/cv2/tensorrt/pycuda/onnxruntime/gradio, ~5-30s)...", flush=True)

def _pip_live(*pkgs, verbose=False, timeout_s=1800):
    # Streams pip's own output live into the cell so progress is never hidden.
    # verbose=True also streams COMPILER output (a source build otherwise shows
    # only a spinner for minutes and looks exactly like a hang).
    t_pip = time.time()
    print(f"  ⬇ pip install {' '.join(pkgs)}", flush=True)
    cmd = [sys.executable, "-m", "pip", "install", "--prefer-binary",
           "--cache-dir", LOCAL_PIP_CACHE, "--retries", "3", "--timeout", "60"]
    if verbose:
        cmd.append("-v")
    cmd.extend(pkgs)
    try:
        r = subprocess.run(cmd, timeout=timeout_s)
    except subprocess.TimeoutExpired:
        print(f"  ⚠ pip timed out after {timeout_s}s — retrying once without cache", flush=True)
        r = subprocess.run([sys.executable, "-m", "pip", "install", *pkgs], timeout=timeout_s)
    if r.returncode != 0:
        print("  ⚠ install with cache/--prefer-binary failed — retrying with pip defaults", flush=True)
        r = subprocess.run([sys.executable, "-m", "pip", "install", *pkgs], timeout=timeout_s)
    print(f"  ⏱ pip step took {_el(t_pip)}", flush=True)
    return r.returncode == 0

_need = []

# numpy (some Colab images pin 1.26, which breaks TensorRT 10.x)
try:
    import numpy as np
    if np.__version__.startswith("1.26"):
        print(f"  numpy {np.__version__} pinned by image — upgrading to 2.x")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "numpy"], capture_output=True)
        _need.append("numpy>=2.0")
    else:
        print(f"  ✓ numpy {np.__version__} (already present)")
except ImportError:
    _need.append("numpy>=2.0")

try:
    import cv2
    print(f"  ✓ opencv {cv2.__version__} (already present)")
except ImportError:
    _need.append("opencv-python-headless")

# TensorRT: only reinstall if missing or wrong major version (saves minutes on re-runs)
_trt_ok = False
try:
    import tensorrt as _trt_probe
    _trt_ok = _trt_probe.__version__.startswith("10.")
    if _trt_ok:
        print(f"  ✓ TensorRT {_trt_probe.__version__} (already present — skipping reinstall)")
except Exception:
    _trt_ok = False
if not _trt_ok:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "tensorrt", "tensorrt-cu12",
                    "tensorrt-cu12-bindings", "tensorrt-cu12-libs"], capture_output=True)
    _need += ["tensorrt-cu12>=10.0,<11", "tensorrt>=10.0,<11"]

_need_pycuda = False
try:
    import pycuda  # noqa: F401
    print("  ✓ pycuda (already present)")
except Exception:
    _need_pycuda = True

try:
    import onnxruntime as _ort_probe
    print(f"  ✓ onnxruntime {_ort_probe.__version__} (already present)")
except Exception:
    # onnxruntime-gpu alone provides the onnxruntime module + CUDAExecutionProvider.
    # Installing plain onnxruntime alongside it makes the two fight over the same
    # package dir (and adds resolver churn) — original Cell 2 ships gpu only.
    _need += ["onnxruntime-gpu"]

try:
    import gradio as _gr_probe
    print(f"  ✓ gradio {_gr_probe.__version__} (already present)")
except Exception:
    _need += ["gradio>=4.0", "pillow", "requests", "tqdm"]

# Installs run in SMALL, ORDERED groups — the same order as the original
# Proteus V3 Cell 1 — NOT one giant combined pass. A single combined pass makes
# pip's resolver backtrack across tensorrt/gradio/onnxruntime version constraints
# (re-downloading candidate wheels for tens of minutes) and hides the multi-minute
# pycuda source compile behind a spinner: both look exactly like a dead hang here.
if _need or _need_pycuda:
    _missing = list(_need) + (["pycuda"] if _need_pycuda else [])
    print(f"\n  Installing {len(_missing)} missing package(s) in small ordered groups: {', '.join(_missing)}", flush=True)
    print("  (each group prints its own timing; pip output streams live below)", flush=True)

    _install_order = [
        ("numpy",           [p for p in _need if p.startswith("numpy")]),
        ("opencv",          [p for p in _need if p.startswith("opencv")]),
        ("tensorrt-cu12",   [p for p in _need if p.startswith("tensorrt-cu12")]),
        ("tensorrt",        [p for p in _need if p.startswith("tensorrt>=")]),
        ("onnxruntime-gpu", [p for p in _need if p.startswith("onnxruntime")]),
        ("UI stack",        [p for p in _need if p.split(">=")[0] in ("gradio", "pillow", "requests", "tqdm")]),
    ]
    _done_pkgs = {p for _, _g in _install_order for p in _g}
    _leftover = [p for p in _need if p not in _done_pkgs]
    if _leftover:
        _install_order.append(("other", _leftover))
    for _label, _pkgs in _install_order:
        if _pkgs:
            print(f"\n  ── {_label}: {' '.join(_pkgs)}", flush=True)
            _pip_live(*_pkgs)

    if _need_pycuda:
        print("\n  ── pycuda (installed LAST, on its own)", flush=True)
        print("     NOTE: pycuda has no prebuilt wheel on PyPI. If the warm cache has no", flush=True)
        print("     wheel matching this runtime's Python version, pip compiles it from source:", flush=True)
        print("     ~2-5 min with COMPILER OUTPUT streaming below (verbose mode is ON).", flush=True)
        print("     gcc lines scrolling = working, NOT stuck. Slowest first-run step by far.", flush=True)
        _pip_live("pycuda", verbose=True, timeout_s=1800)

    for k in list(sys.modules):
        if k == "numpy" or k.startswith("numpy.") or "tensorrt" in k:
            del sys.modules[k]
else:
    print("  ✓ Nothing to install — all dependencies already satisfied")

import numpy as np
try:
    import tensorrt as trt
    print(f"  ✓ TensorRT {trt.__version__}")
except Exception as e:
    print(f"  ⚠ TensorRT unavailable ({e}); ONNX fallback will be used")
import gradio as gr
print(f"  ✓ gradio {gr.__version__}")
try:
    import onnxruntime as _ort2
    print(f"  ✓ onnxruntime {_ort2.__version__} (providers: {', '.join(_ort2.get_available_providers())})")
except Exception as e:
    print(f"  ⚠ onnxruntime unavailable ({e})")
try:
    import pycuda.driver as _pcd  # noqa: F401
    print("  ✓ pycuda importable")
except Exception as e:
    print(f"  ⚠ pycuda unavailable ({e}); TensorRT path will auto-fall back to ONNX")

_r = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
if _r.returncode == 0:
    print(f"  ✓ ffmpeg present ({_r.stdout.splitlines()[0][:60]})")
else:
    print("  ⬇ Installing ffmpeg via apt...")
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=False)
    print("  ✓ ffmpeg installed")
print(f"  ⏱ dependencies step took {_el(_t)}")

# ── [2/4] Model weights — parallel download with a real progress bar ───────
_banner("[2/4] Model weights — parallel download from the MotionSalt GitHub Release")
_t = time.time()
MODELS_DIR = "/content/models"
os.makedirs(MODELS_DIR, exist_ok=True)
RELEASE_BASE = "https://github.com/motionssalt/MotionSalt-Upscaler/releases/download/v15.5-weights"
WEIGHTS = [
    "prob-v3-fgnet-fp16-576x672-1x.trt",
    "prob-v3-fgnet-fp16-576x672-2x.trt",
    "prob-v3-fgnet-fp32-576x672-1x.trt",
    "prob-v3-fgnet-fp32-576x672-2x.trt",
    "prob-v3-fgnet-fp32-576x672-2x.onnx",
]

import requests as _requests
try:
    from tqdm.auto import tqdm as _tqdm
except Exception:
    _pip_live("tqdm")
    from tqdm.auto import tqdm as _tqdm

def _local_ok(fname):
    d = os.path.join(MODELS_DIR, fname)
    return os.path.exists(d) and os.path.getsize(d) > 1_000_000

_todo = [f for f in WEIGHTS if not _local_ok(f)]
for f in WEIGHTS:
    if f not in _todo:
        print(f"  ✓ {f} (already local, {os.path.getsize(os.path.join(MODELS_DIR, f)) // 1048576} MB)")

if _todo:
    print("  Probing file sizes for the progress bar...", end=" ", flush=True)
    _sizes = {}
    for _f in _todo:
        try:
            _h = _requests.head(f"{RELEASE_BASE}/{_f}", allow_redirects=True, timeout=30)
            _sizes[_f] = int(_h.headers.get("content-length", 0))
        except Exception:
            _sizes[_f] = 0
    _total = sum(_sizes.values())
    print(f"done — {len(_todo)} file(s), {_total / 1048576:.0f} MB total")
    print("  ⬇ Downloading with 5 parallel streams (was sequential before):")

    _bar = _tqdm(total=_total or None, unit="B", unit_scale=True,
                 desc="  weights", ncols=90)
    _bar_lock = threading.Lock()

    def _dl_one(fname):
        dst = os.path.join(MODELS_DIR, fname)
        part = dst + ".part"
        with _requests.get(f"{RELEASE_BASE}/{fname}", stream=True, timeout=(30, 120)) as r:
            r.raise_for_status()
            with open(part, "wb") as fh:
                for chunk in r.iter_content(1 << 20):
                    fh.write(chunk)
                    with _bar_lock:
                        _bar.update(len(chunk))
        os.replace(part, dst)
        return fname, os.path.getsize(dst)

    _failed = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as ex:
        _futs = {ex.submit(_dl_one, f): f for f in _todo}
        for _fut in concurrent.futures.as_completed(_futs):
            try:
                _fn, _sz = _fut.result()
                print(f"  ✓ {_fn} ({_sz // 1048576} MB)")
            except Exception as e:
                _failed.append((_futs[_fut], str(e)))
    _bar.close()
    for _fn, _err in _failed:
        print(f"  ✗ FAILED: {_fn} — {_err[:200]}")
    if _failed:
        raise RuntimeError(f"{len(_failed)} weight file(s) failed to download — see errors above")
print(f"  ⏱ weights step took {_el(_t)}")

# ── [3/4] Inference core ───────────────────────────────────────────────────
_banner("[3/4] MotionSalt inference core")
_t = time.time()
CORE_URL = "https://raw.githubusercontent.com/motionssalt/MotionSalt-Upscaler/main/proteus_core.py"
CORE_PATH = "/content/proteus_core.py"
print("  ⬇ Fetching proteus_core.py from the repo (always latest)...")
subprocess.run(["curl", "-sL", "-o", CORE_PATH, CORE_URL], check=True)
if "/content" not in sys.path:
    sys.path.insert(0, "/content")
import importlib, proteus_core
importlib.reload(proteus_core)
from proteus_core import MotionSaltUpscaler
print(f"  ✓ core loaded ({os.path.getsize(CORE_PATH) // 1024} KB) in {_el(_t)}")

_banner("[4/4] Building Gradio UI")
INPUT_DIR, OUTPUT_DIR = "/content/input_uploads", "/content/outputs"
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
import requests as _rq

def _upload_temp_host(path):
    """Auto-upload to a free temp host. Primary: litterbox.catbox.moe (72h,
    1GB, no key). Fallback: tmpfiles.org (~60min). Returns (url, expiry_text)."""
    fname = os.path.basename(path)
    try:
        with open(path, "rb") as f:
            r = _rq.post("https://litterbox.catbox.moe/resources/internals/api.php",
                         data={"reqtype": "fileupload", "time": "72h"},
                         files={"fileToUpload": (fname, f)}, timeout=600)
        if r.status_code == 200 and r.text.strip().startswith("http"):
            return r.text.strip(), "expires in 72 hours (litterbox / catbox temp host)"
    except Exception as e:
        print(f"  litterbox upload failed: {e}")
    try:
        with open(path, "rb") as f:
            r = _rq.post("https://tmpfiles.org/api/v1/upload",
                         files={"file": (fname, f)}, timeout=600)
        if r.status_code == 200:
            url = r.json()["data"]["url"].replace("tmpfiles.org/", "tmpfiles.org/dl/")
            return url, "expires in ~60 minutes (tmpfiles.org)"
    except Exception as e:
        print(f"  tmpfiles upload failed: {e}")
    return None, "temp upload failed — use the direct download button or /content/outputs"

def _fetch_input(uploaded_file, url_text):
    if uploaded_file:
        dst = os.path.join(INPUT_DIR, os.path.basename(uploaded_file))
        shutil.copy(uploaded_file, dst)
        return dst
    if url_text and url_text.strip():
        dst = os.path.join(INPUT_DIR, "url_input_" + str(int(time.time())) +
                           os.path.splitext(url_text.strip().split("?")[0])[1] or ".mp4")
        r = subprocess.run(["curl", "-sL", "--retry", "3", "-o", dst, url_text.strip()],
                           capture_output=True, text=True)
        if r.returncode != 0 or not os.path.exists(dst) or os.path.getsize(dst) < 1000:
            raise RuntimeError(f"Could not download video from URL: {r.stderr[-200:]}")
        return dst
    raise RuntimeError("Provide a video/image file upload OR a direct URL.")

def run_upscale(uploaded_file, url_text, scale, preset, speed_up,
                aad, rn, rd, dh, sh, rc, ro,
                in_vres, in_ires, out_vres, img_fmt, encoder, save_frames, quality,
                progress=gr.Progress(track_tqdm=True)):
    logs = []
    def _log(m):
        logs.append(str(m)); print(m)
    try:
        progress(0.0, desc="Preparing input...")
        input_path = _fetch_input(uploaded_file, url_text)
        _log(f"Input: {input_path}")
        progress(0.05, desc="Loading model...")
        up = MotionSaltUpscaler(
            input_path=input_path, scale=scale, quality_preset=preset, speed_up=speed_up,
            anti_alias_deblur=aad, reduce_noise=rn, recover_details=rd, dehalo=dh,
            sharpen=sh, revert_compression=rc, recover_original=ro,
            input_video_resolution=in_vres, input_image_resolution=in_ires,
            output_video_resize=out_vres, image_format=img_fmt,
            encoder=encoder, save_frames=save_frames, quality=quality,
            verbose=True, log=_log)
        import re as _re
        def _cb(msg):
            # Per-frame messages drive the live progress bar + cell tqdm only
            # (they would flood the log box); milestones go to both bar and log.
            m = _re.search(r"frame (\d+)/(\d+)", str(msg))
            if m:
                frac = 0.1 + 0.8 * (int(m.group(1)) / max(1, int(m.group(2))))
                progress(frac, desc=str(msg))
            else:
                progress(0.1, desc=str(msg)[:60])
                logs.append(str(msg))
        progress(0.1, desc="Upscaling...")
        out_path = up.run(output_dir=OUTPUT_DIR, progress_cb=_cb)
        _log(f"Output written: {out_path}")
        progress(0.9, desc="Uploading to temp host...")
        tmp_url, expiry = _upload_temp_host(out_path)
        if tmp_url:
            _log(f"Temp link: {tmp_url}  ({expiry})")
        else:
            _log(expiry)
        progress(1.0, desc="Done")
        is_video = not any(out_path.lower().endswith(e) for e in
                           [".png", ".tiff", ".tif", ".jpg", ".jpeg"])
        return (out_path if is_video else None,   # video preview
                None if is_video else out_path,   # image preview
                out_path,                          # download button
                "\n".join(logs[-400:]))
    except Exception as e:
        _log(f"ERROR: {e}")
        return None, None, None, "\n".join(logs[-400:])

with gr.Blocks(title="MotionSalt Upscaler") as demo:
    gr.Markdown("# 🧂 MotionSalt Upscaler\nAnime/video upscaling — MotionSalt rebuild of the open-source "
                "**Proteus V3** (V15.5) by its original creator (HF: `legend2008`). "
                "All credit for the model & pipeline goes to them; MotionSalt added this UI + weight re-hosting.")
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📁 Input")
            in_file = gr.File(label="Upload video or image", file_count="single")
            in_url = gr.Textbox(label="...or direct video/image URL", placeholder="https://... (direct link)")
            scale = gr.Dropdown(["2x", "1x"], value="2x", label="Scale")
            preset = gr.Dropdown(["Custom", "V1 Anime 1 Pass", "V2 Anime 2 Pass",
                                  "V3 Anime High Quality 2 Pass"], value="Custom", label="Quality Preset")
            speed_up = gr.Dropdown(["ON", "OFF"], value="ON",
                                   label="Speed (ON = FP16 fast, OFF = FP32 max precision)")
            gr.Markdown("### 🎛️ Custom sliders (active when Preset = Custom)")
            aad = gr.Slider(-100, 100, value=50, step=1, label="Anti-Alias & Deblur")
            rn = gr.Slider(0, 100, value=17, step=1, label="Reduce Noise")
            rd = gr.Slider(0, 100, value=100, step=1, label="Recover Details")
            dh = gr.Slider(0, 100, value=10, step=1, label="Dehalo")
            sh = gr.Slider(0, 100, value=20, step=1, label="Sharpen")
            rc = gr.Slider(0, 100, value=100, step=1, label="Revert Compression")
            ro = gr.Slider(0, 100, value=0, step=1, label="Recover Original Details")
            gr.Markdown("### 📐 Resolutions & output")
            in_vres = gr.Dropdown(["1080p", "1440p", "2160p", "No Resize"], value="1080p",
                                  label="Input Video Resolution (pre-resize)")
            in_ires = gr.Dropdown(["1080p", "1440p", "2160p", "No Resize"], value="1440p",
                                  label="Input Image Resolution")
            out_vres = gr.Dropdown(["Original AI Output", "1080p HD", "1440p 2K", "2160p 4K"],
                                   value="1440p 2K", label="Output Video Resize")
            img_fmt = gr.Dropdown(["PNG 8bit", "PNG 16bit", "TIFF 10bit", "TIFF 12bit", "TIFF 16bit"],
                                  value="PNG 8bit", label="Image Format (image inputs)")
            encoder = gr.Dropdown(["x264 8bit", "x265 8bit", "x265 10bit",
                                   "ProRes 4444 12bit", "FFV1 16bit Lossless"],
                                  value="x265 8bit", label="Video Encoder")
            save_frames = gr.Dropdown(["OFF", "ON"], value="OFF",
                                      label="Save every frame as 16-bit PNG")
            quality = gr.Slider(0, 51, value=16, step=1,
                                label="Quality (lower = better; 13 = lossless)")
            run_btn = gr.Button("▶ Run Upscale", variant="primary", size="lg")
        with gr.Column(scale=1):
            gr.Markdown("### 🎬 Output")
            out_video = gr.Video(label="Video preview")
            out_image = gr.Image(label="Image preview")
            out_download = gr.File(label="⬇ Direct download (served from this Colab)")
            out_log = gr.Textbox(label="Log (includes temp-host link + expiry)", lines=14,
                                 interactive=False)
            gr.Markdown("Files are also kept under `/content/input_uploads` and "
                        "`/content/outputs` — recoverable from Colab's file browser "
                        "even if this tab closes.")
    run_btn.click(run_upscale,
                  inputs=[in_file, in_url, scale, preset, speed_up,
                          aad, rn, rd, dh, sh, rc, ro,
                          in_vres, in_ires, out_vres, img_fmt, encoder, save_frames, quality],
                  outputs=[out_video, out_image, out_download, out_log])

print("\n" + "=" * 70)
print("Launching UI...")
print("  • The inline UI appears below this cell.")
print("  • A public *.gradio.live link is printed next — open it in a NEW TAB for")
print("    the same interface full-screen. It keeps using this Colab's GPU in the")
print("    background for as long as the runtime stays alive.")
print("=" * 70)
print(f"\n⏱ TOTAL setup time: {_el(_T0)}  — share this number when reporting speed")
demo.launch(share=True, debug=True)


## How to use — MotionSalt Upscaler

1. **Runtime → Change runtime type → GPU** (T4 or better), then run the cell above.
2. The cell installs dependencies, downloads the model weights from this repo's
   GitHub Release, and launches the interface. You get an **inline UI** plus a public
   **`*.gradio.live` link** — open that link in a new tab for the full-screen version;
   it still runs on this Colab's GPU.
3. **Upload a video/image** *or* paste a **direct URL**, tune the parameters, hit **Run**.
4. Results: in-UI **preview**, a **direct download** button, and an auto-generated
   **temporary share link** (with its expiry shown in the log). Everything is also
   written to `/content/input_uploads` and `/content/outputs` for recovery from the
   Colab file browser.
5. Presets **V1/V2/V3** override the sliders (1-pass fast, 2-pass, 2-pass high quality).
   Choose **Custom** to drive the seven sliders yourself. No Google Drive option is
   included by design.

### Credit
This notebook is the **MotionSalt fork/rebuild** of the open-source **Proteus V3**
(V15.5) Colab released publicly by its original creator (Hugging Face user
[`legend2008`](https://huggingface.co/legend2008)). The model, inference method,
tiling and encoder recipes are entirely their work — MotionSalt only added the
Gradio interface, removed a hardcoded token from the original setup cell, and
re-hosted the weights as GitHub Release assets. The original files shipped without
an explicit license; this fork's added code is MIT-licensed (see repo `LICENSE`).
